# Categorical Encoding: Count and Frequency Encoding

In **count encoding**, categorical values are replaced by the raw count of observations belonging to that category in the training dataset.

In **frequency encoding** (or percentage encoding), categorical values are replaced by the normalized frequency (proportion) of observations in the training dataset:

$$\text{Frequency}(c) = \frac{\text{Count}(c)}{N_{\text{train}}}$$

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Understand when to use count/frequency encoding as a compact alternative to One-Hot Encoding for **high-cardinality nominal variables**.
2. Prevent **data leakage** by computing counts and frequencies strictly on the training set.
3. Identify and solve the **unseen category problem** in test data (mapping to `NaN`).
4. Understand the risk of **frequency collisions** (distinct categories receiving identical encoded values).
5. Implement reusable, clean frequency encoding in pandas and verify test set outputs.

---
## 1. When to Use (and Caveats of) Frequency Encoding

### When to Use:
- **High-Cardinality Features**: When a nominal feature has dozens or hundreds of categories (e.g., zip codes, cities, vehicle models) where One-Hot Encoding would explode the feature space.
- **Tree-Based Ensembles**: Decision trees and gradient boosting models can easily group rare vs. frequent categories based on frequency values.

### Two Critical Caveats to Address:
1. **Unseen Categories in Test Data**:
   - If a category appears in `X_test` that was never observed in `X_train`, pandas `.map()` will return **`NaN`**!
   - **Solution**: We must explicitly handle missing values after mapping (e.g., filling with `0` or a small default frequency).
2. **Frequency Collisions**:
   - If two completely different categories (e.g., `"Subaru"` and `"Mazda"`) both have exactly 15 cars in the dataset, they both receive the identical value `0.015`. The model cannot distinguish between them based on this feature alone.
3. **No Direct Target Correlation**:
   - Frequency captures how *popular* a category is, not necessarily how predictive it is of the target variable $y$.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10

---
## 2. Loading Data & Splitting Train/Test Sets

We examine high-cardinality nominal features from the Ames House Prices dataset:
- `Neighborhood` (25 distinct categories)
- `Exterior1st` (15 distinct categories)
- `Exterior2nd` (16 distinct categories)

In [2]:
cols_to_use = [
    "Neighborhood",
    "Exterior1st",
    "Exterior2nd",
    "SalePrice",
]

data_path = '../datasets/houseprice.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/cmps460s26/cmps460-content/refs/heads/main/examples/03.dp/datasets/houseprice.csv'

data = pd.read_csv(data_path, usecols=cols_to_use)

# Separate predictors and target
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]

# Train/Test Split (70% train, 30% test) BEFORE computing frequencies
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
X_train.head()

X_train shape: (1022, 3)
X_test shape:  (438, 3)


,Neighborhood,Exterior1st,Exterior2nd
64,CollgCr,VinylSd,VinylSd
682,ClearCr,Wd Sdng,Wd Sdng
960,BrkSide,Wd Sdng,Plywood
1384,Edwards,WdShing,Wd Shng
1100,SWISU,Wd Sdng,Wd Sdng


---
## 3. Method 1: Count Encoding

In count encoding, we count the number of occurrences of each label in `X_train` using `.value_counts()` and map those counts to both train and test sets.

In [3]:
# Calculate raw counts on X_train ONLY
neighborhood_counts = X_train["Neighborhood"].value_counts().to_dict()

print("Top 5 Most Frequent Neighborhoods (Counts):")
for k, v in list(neighborhood_counts.items())[:5]:
    print(f"  {k:12s}: {v} houses")

Top 5 Most Frequent Neighborhoods (Counts):
  NAmes       : 151 houses
  CollgCr     : 105 houses
  OldTown     : 73 houses
  Edwards     : 71 houses
  Sawyer      : 61 houses


In [4]:
# Apply count encoding
X_train_cnt = X_train.copy()
X_test_cnt  = X_test.copy()

# Map counts to train and test
X_train_cnt["Neighborhood_count"] = X_train_cnt["Neighborhood"].map(neighborhood_counts)
X_test_cnt["Neighborhood_count"]  = X_test_cnt["Neighborhood"].map(neighborhood_counts)

# Handle potential unseen categories in test set by filling with 0
X_test_cnt["Neighborhood_count"] = X_test_cnt["Neighborhood_count"].fillna(0).astype(int)

# Inspect sample
X_train_cnt[["Neighborhood", "Neighborhood_count"]].head(6)

,Neighborhood,Neighborhood_count
64,CollgCr,105
682,ClearCr,24
960,BrkSide,41
1384,Edwards,71
1100,SWISU,18
416,Sawyer,61


---
## 4. Method 2: Frequency (Percentage) Encoding

Frequency encoding normalizes the counts by dividing by the total number of training observations ($N_{\text{train}}$). This produces values between $0.0$ and $1.0$ that represent the relative prevalence of each category.

In [5]:
# Calculate relative frequencies on X_train using normalize=True
neighborhood_freqs = X_train["Neighborhood"].value_counts(normalize=True).to_dict()

print("Top 5 Most Frequent Neighborhoods (Frequencies):")
for k, v in list(neighborhood_freqs.items())[:5]:
    print(f"  {k:12s}: {v*100:.2f}% of training set")

Top 5 Most Frequent Neighborhoods (Frequencies):
  NAmes       : 14.77% of training set
  CollgCr     : 10.27% of training set
  OldTown     : 7.14% of training set
  Edwards     : 6.95% of training set
  Sawyer      : 5.97% of training set


In [6]:
# Apply frequency encoding
X_train_freq = X_train.copy()
X_test_freq  = X_test.copy()

X_train_freq["Neighborhood_freq"] = X_train_freq["Neighborhood"].map(neighborhood_freqs)
X_test_freq["Neighborhood_freq"]  = X_test_freq["Neighborhood"].map(neighborhood_freqs)

# Fill unseen test categories with 0.0
X_test_freq["Neighborhood_freq"] = X_test_freq["Neighborhood_freq"].fillna(0.0)

X_train_freq[["Neighborhood", "Neighborhood_freq"]].head(6)

,Neighborhood,Neighborhood_freq
64,CollgCr,0.102740
682,ClearCr,0.023483
960,BrkSide,0.040117
1384,Edwards,0.069472
1100,SWISU,0.017613
416,Sawyer,0.059687


---
## 5. Reusable Multi-Column Frequency Encoding Function

Let's build a clean, modular Python function that computes frequency dictionaries for multiple categorical features on `X_train` and transforms both `X_train` and `X_test` with automatic handling for unseen categories.

In [7]:
def frequency_encode(train_df, test_df, columns):
    """
    Computes frequency mappings on train_df and transforms both train_df and test_df.
    Unseen categories in test_df are safely filled with 0.0.
    """
    train_encoded = train_df.copy()
    test_encoded  = test_df.copy()
    
    mapping_dict = {}
    
    for col in columns:
        # Calculate frequencies strictly on training set
        freq_map = train_df[col].value_counts(normalize=True).to_dict()
        mapping_dict[col] = freq_map
        
        # Map values
        train_encoded[col] = train_encoded[col].map(freq_map)
        test_encoded[col]  = test_encoded[col].map(freq_map).fillna(0.0)
        
    return train_encoded, test_encoded, mapping_dict

# Apply to all categorical columns
cat_cols = ["Neighborhood", "Exterior1st", "Exterior2nd"]
X_train_encoded, X_test_encoded, freq_maps = frequency_encode(X_train, X_test, cat_cols)

print("Preprocessed Training Set (Frequencies replacing original strings):")
X_train_encoded.head()

Preprocessed Training Set (Frequencies replacing original strings):


,Neighborhood,Exterior1st,Exterior2nd
64,0.102740,0.356164,0.345401
682,0.023483,0.144814,0.138943
960,0.040117,0.144814,0.109589
1384,0.069472,0.020548,0.028376
1100,0.017613,0.144814,0.138943


In [8]:
# Verify that test set contains zero nulls after transformation
print(f"Missing values in X_test after frequency encoding: {X_test_encoded.isnull().sum().sum()}")
X_test_encoded.head()

Missing values in X_test after frequency encoding: 0


,Neighborhood,Exterior1st,Exterior2nd
529,0.034247,0.144814,0.003914
491,0.147750,0.144814,0.138943
459,0.040117,0.135029,0.133072
279,0.023483,0.084149,0.109589
655,0.009785,0.149706,0.007828


---
## 6. Investigating the Frequency Collision Risk

Let's check if any distinct categories share the exact same frequency in `Exterior1st`.

In [9]:
# Check value counts of frequencies to detect collisions
freq_series = pd.Series(freq_maps['Exterior1st'])
duplicates = freq_series[freq_series.duplicated(keep=False)]

print(f"Total categories in Exterior1st: {len(freq_series)}")
print(f"Categories with shared frequencies (collisions): {len(duplicates)}")
if len(duplicates) > 0:
    print("\nColliding categories:")
    for cat, freq in duplicates.items():
        print(f"  - {cat:15s}: {freq:.4f}")

Total categories in Exterior1st: 15
Categories with shared frequencies (collisions): 4

Colliding categories:
  - AsphShn        : 0.0010
  - BrkComm        : 0.0010
  - ImStucc        : 0.0010
  - CBlock         : 0.0010


### Pedagogical Insight:
Notice how rare categories that appeared the same number of times in the training data receive the exact same frequency value. If distinguishing between those specific rare categories is vital to the downstream task, consider **One-Hot Encoding** or **Target Encoding** instead.

---
## 7. Summary & Best Practices Checklist

| Consideration | Recommendation |
| :--- | :--- |
| **When to Use** | High-cardinality nominal features where One-Hot Encoding would create too many columns. |
| **Data Leakage** | Compute counts or percentages strictly on `X_train`. Never compute on full dataset or test set! |
| **Unseen Categories** | Always chain with `.fillna(0.0)` for test data so unseen categories don't produce unexpected NaNs. |
| **Collision Risk** | Distinct categories with identical counts will be mapped to the exact same value. |
| **Algorithm Match** | Excellent for **Decision Trees**, **Random Forests**, and **Gradient Boosting** (LightGBM/XGBoost). |
